# CNN full held-out segment evaluation

This notebook preserves the useful full-observation review from the legacy workflow without treating subset-local row positions as full-dataset identifiers. It evaluates every labelled row of the `segment_index` values held out by the corrected group split. The CNN checkpoint and operating threshold have already been selected by the baseline runner; this notebook performs no fitting, model choice, or threshold choice.

The result is a held-out-segment evaluation within the same observation, not transfer to an independent observation.

In [ ]:
from pathlib import Path
import json
import subprocess
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader
from tqdm import tqdm

REPO_ROOT = next((parent for parent in (Path.cwd().resolve(), *Path.cwd().resolve().parents) if (parent / 'src' / 'rfimt').is_dir()), None)
if REPO_ROOT is None:
    raise RuntimeError('Open this notebook from a directory inside the rfimt repository.')
if str(REPO_ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / 'src'))

from rfimt.experiments import load_experiment_spec, make_run_manifest, write_run_manifest
from rfimt.metrics import eval_binary
from rfimt.models import CNN1DRFI256Logits
from rfimt.diagnostics import build_segment_review_queue, save_spectrogram_mask_pair
from rfimt.training import ProfileDataset, normalize_profile_segments

CONFIG_PATH = REPO_ROOT / 'configs/experiments/b0531_cnn_full_heldout_segments_legacy_max_v1.json'
spec = load_experiment_spec(CONFIG_PATH)
device_name = spec['evaluation']['device']
if device_name not in {'cpu', 'cuda'}:
    raise ValueError("evaluation.device must be 'cpu' or 'cuda'.")
if device_name == 'cuda' and not torch.cuda.is_available():
    raise RuntimeError('This evaluation is configured for CUDA, but CUDA is unavailable.')
device = torch.device(device_name)
print(f'Repository: {REPO_ROOT}')
print(f'Device: {device}')

## Read the frozen baseline decision

The source manifest is the authority for the selected threshold. Requiring matching model and normalization settings prevents a checkpoint from being evaluated under a different representation.

In [ ]:
baseline_manifest_path = Path(spec['model']['source_manifest'])
checkpoint_path = Path(spec['model']['source_checkpoint'])
if not baseline_manifest_path.is_file() or not checkpoint_path.is_file():
    raise FileNotFoundError('Run the CNN baseline first and provide its manifest and checkpoint paths.')

with baseline_manifest_path.open('r', encoding='utf-8') as handle:
    baseline_manifest = json.load(handle)
baseline_spec = baseline_manifest['spec']
if baseline_manifest['experiment_id'] != spec['model']['source_experiment_id']:
    raise ValueError('Source manifest experiment_id does not match this evaluator configuration.')
manifest_checkpoint = Path(baseline_manifest['artifacts']['checkpoint']).resolve()
if manifest_checkpoint != checkpoint_path.resolve():
    raise ValueError('Source manifest checkpoint does not match this evaluator configuration.')
threshold = float(baseline_manifest['metrics']['validation']['threshold'])

for section, key in (
    ('preprocessing', 'normalization_variant'),
    ('representation', 'kind'),
    ('model', 'factory_name'),
    ('model', 'parameters'),
):
    if baseline_spec[section][key] != spec[section][key]:
        raise ValueError(f'Baseline and evaluator disagree on {section}.{key}.')
if baseline_spec['selection']['threshold_source'] != 'validation':
    raise ValueError('The source threshold was not selected from validation data.')

print(f"Baseline run: {baseline_manifest['experiment_id']}")
print(f'Frozen validation threshold: {threshold:.2f}')

## Recover complete held-out segments

The corrected split stores positions within its compact subset metadata. Those positions are used only to recover the test `segment_index` values. The full observation is then selected by these group identifiers, so no train or validation segment can enter this evaluation.

In [ ]:
GROUP_COLUMN = spec['split']['group_column']
group_dir = Path(spec['dataset']['group_dataset_dir'])
subset_meta = pd.read_csv(group_dir / 'subset_channels_meta.csv').fillna('None')
with np.load(spec['split']['indices_path']) as split_file:
    split_indices = {name: np.asarray(split_file[name], dtype=int) for name in ('train', 'val', 'test')}

split_groups = {name: set(subset_meta.iloc[indices][GROUP_COLUMN].tolist()) for name, indices in split_indices.items()}
if split_groups['train'] & split_groups['val'] or split_groups['train'] & split_groups['test'] or split_groups['val'] & split_groups['test']:
    raise RuntimeError('Corrected split contains overlapping segment_index values.')
test_groups = split_groups['test']
if not test_groups:
    raise RuntimeError('The corrected split has no test segments.')

full_meta = pd.read_csv(spec['dataset']['full_metadata_path']).fillna('None').copy()
full_meta['source_row_index'] = np.arange(len(full_meta), dtype=int)
full_array = np.load(spec['dataset']['full_array_path'], mmap_mode='r')
if len(full_meta) != len(full_array):
    raise ValueError('Full metadata and full array have different row counts.')

evaluation_meta = full_meta.loc[full_meta[GROUP_COLUMN].isin(test_groups)].copy()
if evaluation_meta.empty:
    raise RuntimeError('No full-observation rows match the held-out test segments.')
if set(evaluation_meta[GROUP_COLUMN]) != test_groups:
    raise RuntimeError('At least one held-out segment is missing from the full metadata.')

allowed_labels = {spec['labels']['positive'], spec['labels']['ordinary_negative'], spec['labels']['hard_negative']}
unknown_labels = sorted(set(evaluation_meta['label']) - allowed_labels)
if unknown_labels:
    raise ValueError(f'Unexpected labels in held-out full rows: {unknown_labels}')
evaluation_meta['target'] = evaluation_meta['label'].eq(spec['labels']['positive']).astype(int)
print(f'Held-out segments: {len(test_groups)}')
print(evaluation_meta['label'].value_counts().rename('rows'))

## Infer once on all held-out rows

Inference is batched to keep the complete review practical on the server. Row order remains fixed, so scores can be joined back to metadata without using ambiguous identifiers.

In [ ]:
model = CNN1DRFI256Logits(**spec['model']['parameters']).to(device)
checkpoint = torch.load(checkpoint_path, map_location=device)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

source_indices = evaluation_meta['source_row_index'].to_numpy(dtype=int)
profiles = np.asarray(full_array[source_indices])
targets = evaluation_meta['target'].to_numpy(dtype=np.float32)
normalization = spec['preprocessing']['normalization_variant']
if normalization == 'zscore_per_segment':
    # The physical segment, not the inference batch, defines this scale.
    profiles = normalize_profile_segments(profiles, evaluation_meta[GROUP_COLUMN].to_numpy())
    normalization = 'none'
dataset = ProfileDataset(profiles, targets, normalization=normalization)
loader = DataLoader(dataset, batch_size=spec['evaluation']['inference_batch_size'], shuffle=False)

score_chunks = []
target_chunks = []
with torch.no_grad():
    for batch_profiles, batch_targets in tqdm(loader, desc='Held-out CNN inference', mininterval=5.0, miniters=10):
        logits = model(batch_profiles.to(device))
        score_chunks.append(torch.sigmoid(logits).cpu().numpy().reshape(-1))
        target_chunks.append(batch_targets.numpy().reshape(-1))
scores = np.concatenate(score_chunks)
inferred_targets = np.concatenate(target_chunks).astype(int)
if not np.array_equal(inferred_targets, evaluation_meta['target'].to_numpy(dtype=int)):
    raise RuntimeError('Inference loader changed the held-out row order.')

evaluation_meta['score'] = scores
evaluation_meta['prediction'] = (scores >= threshold).astype(int)
row_metrics = eval_binary(inferred_targets, scores, threshold=threshold)
row_metrics

## Write audit tables

The overall metric answers the original binary task. The label breakdown keeps clean `None` and excluded `NoneWNBRFI` distinguishable, which is necessary when interpreting false positives. Per-segment counts make it possible to trace any aggregate result back to a concrete observation interval.

In [ ]:
run_dir = Path(spec['outputs']['run_directory'])
if run_dir.exists():
    raise FileExistsError(f'Refusing to overwrite existing run directory: {run_dir}')
run_dir.mkdir(parents=True)

def count_errors(frame):
    target = frame['target'].to_numpy(dtype=int)
    prediction = frame['prediction'].to_numpy(dtype=int)
    return {
        'n_rows': int(len(frame)),
        'TN': int(((target == 0) & (prediction == 0)).sum()),
        'FP': int(((target == 0) & (prediction == 1)).sum()),
        'FN': int(((target == 1) & (prediction == 0)).sum()),
        'TP': int(((target == 1) & (prediction == 1)).sum()),
    }

label_rows = []
for label, frame in evaluation_meta.groupby('label', sort=True):
    counts = count_errors(frame)
    counts['label'] = label
    counts['mean_score'] = float(frame['score'].mean())
    counts['positive_prediction_fraction'] = float(frame['prediction'].mean())
    counts['false_positive_rate'] = float(counts['FP'] / (counts['TN'] + counts['FP'])) if counts['TN'] + counts['FP'] else None
    counts['false_negative_rate'] = float(counts['FN'] / (counts['FN'] + counts['TP'])) if counts['FN'] + counts['TP'] else None
    label_rows.append(counts)
label_table = pd.DataFrame(label_rows).sort_values('label').reset_index(drop=True)

segment_rows = []
for segment_index, frame in evaluation_meta.groupby(GROUP_COLUMN, sort=True):
    counts = count_errors(frame)
    counts[GROUP_COLUMN] = segment_index
    counts['labels_present'] = '|'.join(sorted(frame['label'].unique()))
    counts['mean_score'] = float(frame['score'].mean())
    counts['positive_prediction_fraction'] = float(frame['prediction'].mean())
    # Keep the operating point with each segment so review selection is auditable.
    counts['threshold'] = float(threshold)
    segment_rows.append(counts)
segment_table = pd.DataFrame(segment_rows).sort_values(GROUP_COLUMN).reset_index(drop=True)

evaluation_meta.to_csv(run_dir / 'full_heldout_row_scores.csv', index=False)
label_table.to_csv(run_dir / 'label_breakdown.csv', index=False)
segment_table.to_csv(run_dir / 'full_heldout_segment_metrics.csv', index=False)
label_table

## Create a bounded visual-review queue

The queue does not alter metrics. It selects a broader set of failures, successful recoveries, clean references and threshold-borderline cases so that visual inspection can explain both errors and apparently good aggregate metrics.

In [ ]:
n_per_category = int(spec['evaluation']['review_segments_per_category'])
review_queue = build_segment_review_queue(segment_table, n_per_category)
review_queue.to_csv(run_dir / 'visual_review_queue.csv', index=False)

diagnostic_dir = run_dir / 'diagnostics'
diagnostic_dir.mkdir()
for record in tqdm(review_queue.itertuples(index=False), total=len(review_queue), desc='Writing diagnostics', mininterval=5.0, miniters=2):
    segment_index = record.segment_index
    frame = evaluation_meta.loc[evaluation_meta[GROUP_COLUMN].eq(segment_index)].copy()
    sort_column = 'channel_index' if 'channel_index' in frame.columns else 'source_row_index'
    frame = frame.sort_values(sort_column)
    segment_profiles = np.asarray(full_array[frame['source_row_index'].to_numpy(dtype=int)])
    counts = segment_table.loc[segment_table[GROUP_COLUMN].eq(segment_index)].iloc[0]
    title = (f'Held-out segment {segment_index} | {record.review_reason} | '
             f'TN={counts.TN}, FP={counts.FP}, FN={counts.FN}, TP={counts.TP} | threshold={threshold:.2f}')
    save_spectrogram_mask_pair(
        segment_profiles,
        frame['prediction'].to_numpy(dtype=bool),
        diagnostic_dir / f'segment_{segment_index}_{record.review_reason}.png',
        title,
    )


review_queue

## Record the evaluation

This manifest records the source baseline and frozen threshold alongside the full held-out-segment result. It makes clear that the test did not select the operating point.

In [ ]:
def json_ready(value):
    if isinstance(value, dict):
        return {key: json_ready(item) for key, item in value.items()}
    if isinstance(value, (np.floating, np.integer)):
        return value.item()
    if isinstance(value, float) and not np.isfinite(value):
        return None
    return value

code_revision = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=REPO_ROOT, text=True).strip()
manifest = make_run_manifest(
    spec,
    code_revision=code_revision,
    metrics=json_ready({
        'full_heldout_rows': row_metrics,
        'n_heldout_segments': int(len(test_groups)),
        'frozen_validation_threshold': threshold,
        'source_baseline_experiment_id': baseline_manifest['experiment_id'],
    }),
    artifacts={
        'source_baseline_manifest': str(baseline_manifest_path),
        'source_checkpoint': str(checkpoint_path),
        'row_scores': str(run_dir / 'full_heldout_row_scores.csv'),
        'label_breakdown': str(run_dir / 'label_breakdown.csv'),
        'segment_metrics': str(run_dir / 'full_heldout_segment_metrics.csv'),
        'visual_review_queue': str(run_dir / 'visual_review_queue.csv'),
        'diagnostic_directory': str(diagnostic_dir),
    },
    notes=[
        'All rows of group-held-out segments were evaluated.',
        'The threshold was reused from the source run validation universe and was not selected here.',
        'This is not an independent-observation transfer evaluation.',
    ],
)
write_run_manifest(run_dir / 'run_manifest.json', manifest)
print(f'Wrote full held-out evaluation to {run_dir}')
print(json.dumps(json_ready(row_metrics), indent=2, sort_keys=True))